# COSMOS77-ex05 — AirLLM Benchmark (Qwen2.5-14B on a free T4)

**Orchestration of AI Agents (203.3763) · HW5** — Abdallah Khaldi & Tasneem Natour.

Runs the WHOLE experiment headless on a free Kaggle **T4 (16 GB)** and writes every
measured number to **`/kaggle/working/results/*.json`** (the captured ledger):

1. **Hardware** — capture the T4 spec + the param→memory math (D1).
2. **FP16 baseline** — load `Qwen/Qwen2.5-14B-Instruct` (~29 GB) naively →
   expected `torch.cuda.OutOfMemoryError` (bottleneck = **VRAM capacity**) (D2).
3. **AirLLM** — the SAME model runs layer-by-layer (`mmap`, "layer = page") (D3).
4. **Quantization sweep** — AirLLM at **8bit** then **4bit** (bitsandbytes) (D4).

All logic lives in our tested `cosmos77_ex05` package (pip-installed from the public
repo); the cells just call the SDK. The rendered outputs (the OOM traceback, the
AirLLM output, the summary table) are the report's "screenshots". **The grade is the
analysis** — numbers are measured, never fabricated; a partial/negative result is OK.

> Settings: **Accelerator = GPU T4 ×1**, **Internet = On**. Qwen2.5-14B is ungated.
> **Disk:** `/kaggle/temp` (download + shards) and `/kaggle/working` share ONE
> ephemeral overlay (~58–73 GB). We keep the download once, clear shards between
> scenarios, and save only the small JSONs — each stage is wrapped so one failure
> cannot strand the ledger.

In [ ]:
# Cell 1 — environment. Big download + shards go to scratch; the ledger goes to the
# captured output dir; install the experiment stack + our tested package.
import os, shutil, subprocess, sys

SCRATCH = "/kaggle/temp"
RESULTS = "/kaggle/working/results"
os.environ["HF_HOME"] = f"{SCRATCH}/hf"          # 29 GB model cache -> scratch, not output
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
for d in (f"{SCRATCH}/hf", f"{SCRATCH}/shards", RESULTS):
    os.makedirs(d, exist_ok=True)

free_gb = shutil.disk_usage(SCRATCH).free / 1e9
print(f"free scratch disk: {free_gb:.1f} GB")
if free_gb < 55:
    print("WARNING: < 55 GB free — the 14B FP16 shard step (download ~29 GB + shards "
          "~29 GB) may overflow. If a stage dies with 'No space left', switch "
          "config/setup.json model_id to Qwen/Qwen2.5-7B-Instruct (and say so in the report).")

# Optional HF token (Qwen2.5 is ungated; only avoids rate limits).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret (fine — Qwen2.5-14B is ungated)")

def sh(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

# torch / transformers / accelerate are preinstalled on Kaggle; add the rest.
sh([sys.executable, "-m", "pip", "install", "-q", "-U", "airllm", "bitsandbytes", "accelerate"])
if not os.path.isdir("/kaggle/working/repo"):
    sh(["git", "clone", "-q", "https://github.com/AbdallahKhaldi/COSMOS77-ex05.git", "/kaggle/working/repo"])
sh([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/repo"])
print("environment ready")

In [ ]:
# Cell 2 — SDK + robust helpers. The ledger is written DIRECTLY to /kaggle/working/
# results so every record survives even if a later stage crashes.
import json, traceback
os.chdir("/kaggle/working/repo")
from cosmos77_ex05.sdk.sdk import SDK
sdk = SDK(results_dir=RESULTS)

def clear_shards():
    shutil.rmtree(f"{SCRATCH}/shards", ignore_errors=True)
    os.makedirs(f"{SCRATCH}/shards", exist_ok=True)
    print(f"shards cleared; free disk: {shutil.disk_usage(SCRATCH).free/1e9:.1f} GB")

def run_stage(name, fn):
    print(f"\n===== {name} =====")
    try:
        out = fn(); print(f"[OK] {name}"); return out
    except Exception:
        traceback.print_exc()
        print(f"[FAIL] {name} — continuing; the partial ledger stays valid (D15 honesty).")
        return None

hw = run_stage("hardware", sdk.capture_hardware)
print(json.dumps(hw, indent=2) if hw else "hardware capture failed")

## FP16 baseline — the naive direct load that must fail (D2)

Loading a ~29 GB FP16 model with `device_map="cuda"` onto a 16 GB T4 is expected to raise `torch.cuda.OutOfMemoryError`. We catch it and record requested-vs-available VRAM + the param→memory math. The bottleneck is **memory capacity**, not compute.

In [ ]:
# Cell 3 — FP16 baseline (expected OOM). Downloads ~29 GB first, then OOMs on load.
baseline = run_stage("fp16_baseline", sdk.run_baseline)
print(json.dumps(baseline, indent=2) if baseline else "no baseline record")

## AirLLM — the SAME model runs, layer by layer (D3, "layer = page")

AirLLM keeps only the active transformer layer in VRAM and streams the rest from per-layer SafeTensors shards (`mmap`) — the OS virtual-memory/Paging analogy. It runs the 29 GB model in 16 GB at the cost of latency (disk BW ≪ HBM BW → ~1–3 tok/s).

In [ ]:
# Cell 4 — AirLLM FP16 run (slow: shards then generates). Records airllm_none.
clear_shards()
airllm = run_stage("airllm_none", sdk.run_airllm)
if airllm:
    print({k: v for k, v in airllm.items() if k != "output"})
    print("\nOUTPUT:\n", (airllm.get("output", "") or "")[:600])

## Quantization sweep — AirLLM at 8bit then 4bit (D4)

bitsandbytes quantization (CUDA-only) lowers bytes/param: Q8 ≈ 14.7 GB, Q4 ≈ 7.4 GB. Each level is run SEPARATELY (shards cleared between) so a failure in one still leaves the others in the ledger; the output is captured verbatim for the accuracy "red line".

In [ ]:
# Cell 5 — 8bit. Records airllm_8bit.
clear_shards()
q8 = run_stage("airllm_8bit", lambda: sdk.run_quant_sweep(["8bit"]))
if q8:
    m = q8["8bit"]; keep = ("compression", "throughput_tok_s", "ttft_s", "tpot_ms", "peak_vram_gb")
    print({k: m.get(k) for k in keep}); print("   output:", (m.get("output", "") or "")[:200])

In [ ]:
# Cell 6 — 4bit. Records airllm_4bit.
clear_shards()
q4 = run_stage("airllm_4bit", lambda: sdk.run_quant_sweep(["4bit"]))
if q4:
    m = q4["4bit"]; keep = ("compression", "throughput_tok_s", "ttft_s", "tpot_ms", "peak_vram_gb")
    print({k: m.get(k) for k in keep}); print("   output:", (m.get("output", "") or "")[:200])

## Summary — the measured ledger

Read every `results/*.json` already in `/kaggle/working/results` (the downloadable output) and print the comparison table across all scenarios that completed.

In [ ]:
# Cell 7 — summary table from whatever completed (robust to partial runs).
import pandas as pd
ledger = sdk.ledger()
print("scenarios captured:", sorted(ledger))
rows = []
for scenario, m in sorted(ledger.items()):
    if scenario == "hardware":
        continue
    rows.append({
        "scenario": scenario, "success": m.get("success"),
        "tok_per_s": m.get("throughput_tok_s"), "ttft_s": m.get("ttft_s"),
        "tpot_ms": m.get("tpot_ms"), "peak_vram_gb": m.get("peak_vram_gb"),
        "peak_ram_gb": m.get("peak_ram_gb"),
    })
print()
print(pd.DataFrame(rows).to_string(index=False) if rows else "no scenario metrics captured")